In [75]:
!pip install --quiet riskfolio-lib

In [76]:
!pip install yfinance --quiet

In [77]:
from datetime import timedelta
import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
import riskfolio as rp

MARKET = "^TWII"  
CACHE = {}

companies = {
    "Electronic": ["2330.TW", "2317.TW", "2454.TW"],
    "Construction": ["2404.TW", "5536.TWO", "1101.TW"],
    "Insurance": ["2850.TW", "2881.TW", "2882.TW"],  
    "Energy": ["8926.TW", "6505.TW", "6806.TW"],
}

def fetch_series(ticker, start, end, auto_adjust=True):
    key = (ticker, pd.to_datetime(start).date(), pd.to_datetime(end).date(), auto_adjust)
    if key in CACHE:
        return CACHE[key]
    df = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=auto_adjust, actions=False)
    if df is None or df.empty:
        df = pd.DataFrame(columns=["Close"])
    else:
        if "Close" not in df.columns and "Adj Close" in df.columns:
            df = df.rename(columns={"Adj Close":"Close"})
        df = df[["Close"]].copy()
        df["ret"] = df["Close"].pct_change()
        df.index = pd.to_datetime(df.index)
        df = _make_index_tz_naive(df)  # <-- make tz-naive here
    CACHE[key] = df
    return df

def trading_window(df, event_date, pre=10, post=10):
    """
    Return exactly pre+post+1 trading rows centered on the first trading day >= event_date.
    If event_date is a holiday/weekend, '0' is the next trading day.
    """
    df = _make_index_tz_naive(df)  # <-- ensure tz-naive index
    if df.empty:
        return pd.DataFrame(columns=["date","ret","td"])
    idx = df.index
    ed = pd.to_datetime(event_date).tz_localize(None)  # <-- tz-naive event_date
    mask = idx >= ed
    if not mask.any():
        return pd.DataFrame(columns=["date","ret","td"])
    anchor_pos = int(np.argmax(mask))
    lo = max(0, anchor_pos - pre)
    hi = min(len(idx)-1, anchor_pos + post)
    take = df.iloc[lo:hi+1].copy()
    take["td"] = np.arange(lo, hi+1) - anchor_pos
        # make the index a proper column named 'date' (works for all pandas versions)
    take = take.reset_index()
    # ensure the first column is renamed to 'date'
    first_col = take.columns[0]
    if first_col != "date":
        take = take.rename(columns={first_col: "date"})
    return take[["date", "ret", "td"]]


def fit_market_model(ticker, event_date):
    # pull long history for estimation
    start = pd.to_datetime(event_date) - pd.Timedelta(days=365)
    end   = pd.to_datetime(event_date) + pd.Timedelta(days=30)
    s = fetch_series(ticker, start, end)
    m = fetch_series(MARKET, start, end)
    if s.empty or m.empty:
        return None

    s = _make_index_tz_naive(s)
    m = _make_index_tz_naive(m)

    df = s[["ret"]].rename(columns={"ret":"r_s"}).join(
        m[["ret"]].rename(columns={"ret":"r_m"}), how="inner"
    ).dropna()
    if df.empty:
        return None

    ed = pd.to_datetime(event_date).tz_localize(None)  # <-- tz-naive
    mask = df.index >= ed
    if not mask.any():
        return None

    anchor_pos = int(np.argmax(mask))
    df = df.reset_index()
    df["td"] = np.arange(len(df)) - anchor_pos
    est = df[(df["td"] <= -20) & (df["td"] >= -120)]
    if est.empty:
        return None

    X = np.column_stack([np.ones(len(est)), est["r_m"].values])
    y = est["r_s"].values
    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]  # [alpha, beta]
    return {"alpha": beta_hat[0], "beta": beta_hat[1], "df": df}


def event_metrics_for(ticker, event_date, sector, event_id, magnitude, car_windows=(3,5,10)):
    # trading-day event slice for stock & market
    start = pd.to_datetime(event_date) - pd.Timedelta(days=365)
    end   = pd.to_datetime(event_date) + pd.Timedelta(days=60)
    s = fetch_series(ticker, start, end)
    m = fetch_series(MARKET, start, end)
    if s.empty or m.empty:
        return []

    s = _make_index_tz_naive(s)
    m = _make_index_tz_naive(m)

    sw = trading_window(s, event_date, pre=max(car_windows), post=max(car_windows))
    mw = trading_window(m, event_date, pre=max(car_windows), post=max(car_windows))
    if sw.empty or mw.empty:
        return []

    mm = fit_market_model(ticker, event_date)
    if mm is None:
        return []
    alpha, beta = mm["alpha"], mm["beta"]

    df = sw.merge(
        mw[["date","ret","td"]].rename(columns={"ret":"r_m"}),
        on=["date","td"], how="inner"
    ).rename(columns={"ret":"r_s"}).dropna()
    if df.empty:
        return []

    df["ar"] = df["r_s"] - (alpha + beta*df["r_m"])
    out = [{
        "ticker": ticker, "sector": sector, "event_id": event_id, "event_date": pd.to_datetime(event_date),
        "magnitude": magnitude, "alpha": alpha, "beta": beta,
        "ar_0": float(df.loc[df["td"]==0, "ar"].iloc[0]) if (df["td"]==0).any() else np.nan,
        **{f"car_0_to_{k}": float(df.loc[(df["td"]>=0) & (df["td"]<=k), "ar"].sum()) for k in car_windows}
    }]
    return out

    
def _make_index_tz_naive(df: pd.DataFrame) -> pd.DataFrame:
    """Convert any tz-aware DatetimeIndex to tz-naive (UTC->naive). Safe if already naive/empty."""
    if df is None or df.empty:
        return df
    idx = pd.to_datetime(df.index)
    # if tz-aware, convert to UTC then strip tz
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    df.index = idx
    return df


In [78]:
# Example: replace with your USGS-fed events
earthquakes = pd.DataFrame({
    "event_id": [1],
    "date": [pd.Timestamp("2024-04-03")],
    "magnitude": [7.2],
})

results = []
for sector, tickers in companies.items():
    for t in tickers:
        for _, ev in earthquakes.iterrows():
            rows = event_metrics_for(
                ticker=t,
                event_date=ev["date"],
                sector=sector,
                event_id=ev["event_id"],
                magnitude=ev["magnitude"],
                car_windows=(3,5,10)
            )
            results.extend(rows)

event_metrics = pd.DataFrame(results)
event_metrics.sort_values(["event_id","sector","ticker"], inplace=True)
event_metrics.head(10)

,ticker,sector,event_id,event_date,magnitude,alpha,beta,ar_0,car_0_to_3,car_0_to_5,car_0_to_10
5,1101.TW,Construction,1,2024-04-03,7.2,-0.001239,0.508699,0.008610,0.007121,0.015528,0.055723
3,2404.TW,Construction,1,2024-04-03,7.2,0.003371,0.714496,-0.014980,-0.019088,0.020697,-0.009312
4,5536.TWO,Construction,1,2024-04-03,7.2,0.001125,0.613437,0.006263,0.049443,0.078858,-0.004495
1,2317.TW,Electronic,1,2024-04-03,7.2,-0.000535,0.508867,-0.007759,-0.065815,-0.083866,-0.074096
0,2330.TW,Electronic,1,2024-04-03,7.2,0.000587,1.755449,-0.003655,0.012899,0.020303,0.026792
2,2454.TW,Electronic,1,2024-04-03,7.2,0.002940,1.177575,-0.016199,-0.001363,-0.066000,-0.156574
10,6505.TW,Energy,1,2024-04-03,7.2,-0.002039,0.592545,0.005238,0.009941,0.026949,0.056901
11,6806.TW,Energy,1,2024-04-03,7.2,0.004027,0.128358,-0.004533,-0.030068,-0.042866,-0.101308
9,8926.TW,Energy,1,2024-04-03,7.2,0.000670,0.365209,-0.015850,-0.007637,0.002696,0.010212
6,2850.TW,Insurance,1,2024-04-03,7.2,0.001459,0.161296,-0.016129,0.035364,0.042743,0.068873


In [79]:
CAR_WINDOWS = (3,5,10)

earthquakes_recent = pd.DataFrame([
    {"event_id": 1, "date": "2018-02-06", "magnitude": 6.4},
    {"event_id": 2, "date": "2019-04-18", "magnitude": 6.1},
    {"event_id": 3, "date": "2021-10-24", "magnitude": 6.5},
    {"event_id": 4, "date": "2022-09-18", "magnitude": 6.9},
    {"event_id": 5, "date": "2024-04-03", "magnitude": 7.2},
])
earthquakes_recent["date"] = pd.to_datetime(earthquakes_recent["date"]).dt.tz_localize(None)

# REBUILD event_metrics using ONLY earthquakes_recent
results, skipped = [], []
for sector, tickers in companies.items():
    for t in tickers:
        for _, ev in earthquakes_recent.iterrows():
            rows = event_metrics_for(
                ticker=t,
                event_date=ev["date"],
                sector=sector,
                event_id=ev["event_id"],
                magnitude=ev["magnitude"],
                car_windows=CAR_WINDOWS
            )
            if not rows:
                skipped.append((t, sector, int(ev["event_id"]), pd.to_datetime(ev["date"]).date()))
                continue
            results.extend(rows)

event_metrics = pd.DataFrame(results)
print("rebuilt event_metrics rows:", len(event_metrics))
print("unique events now:", event_metrics["event_id"].nunique() if not event_metrics.empty else 0)

# --- Coverage matrix (which ticker has data for which event) ---
if not event_metrics.empty:
    cov = (event_metrics
           .assign(has=1)
           .pivot_table(index="ticker", columns="event_id", values="has", fill_value=0))
    print("\nCoverage matrix (1=has CAR for event):")
    display(cov)

# --- Rank tickers by average CAR over post-2018 events ---
def rank_tickers_by_avg_car(event_metrics: pd.DataFrame,
                            earthquakes: pd.DataFrame,
                            window: str = "car_0_to_5",
                            mode: str = "zero_impute",
                            ascending: bool = False) -> pd.DataFrame:
    assert window in {"car_0_to_3","car_0_to_5","car_0_to_10"}
    ALL_EV = earthquakes["event_id"].tolist()
    ALL_TK = sorted(event_metrics["ticker"].unique())

    wide = (
        event_metrics
        .set_index(["ticker","event_id"])[[window]]
        .reindex(pd.MultiIndex.from_product([ALL_TK, ALL_EV], names=["ticker","event_id"]))
    )
    n_obs = wide.groupby(level="ticker").apply(lambda df: df[window].notna().sum()).rename("n_obs")

    if mode == "zero_impute":
        agg = wide.fillna(0).groupby(level="ticker").mean().rename(columns={window: f"avg_{window}"})
        agg[f"denom_{window}"] = len(ALL_EV)
    elif mode == "available":
        agg = wide.groupby(level="ticker").mean().rename(columns={window: f"avg_{window}"})
        agg[f"denom_{window}"] = n_obs.values
    else:
        raise ValueError("mode must be 'zero_impute' or 'available'")

    out = agg.join(n_obs).sort_values(f"avg_{window}", ascending=ascending).reset_index()
    return out

rank_zero_recent = rank_tickers_by_avg_car(event_metrics, earthquakes_recent, window="car_0_to_5", mode="zero_impute", ascending=False)
rank_avail_recent = rank_tickers_by_avg_car(event_metrics, earthquakes_recent, window="car_0_to_5", mode="available",   ascending=False)

print("\nTop by avg 5-day CAR (zero-impute, denominator = all post-2018 events):")
display(rank_zero_recent.head(12))

print("Top by avg 5-day CAR (available-only, denominator = observed post-2018 events):")
display(rank_avail_recent.head(12))

$6806.TW: possibly delisted; no price data found  (1d 2017-02-06 00:00:00 -> 2018-04-07 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1486310400, endDate = 1523030400")
$6806.TW: possibly delisted; no price data found  (1d 2018-04-18 00:00:00 -> 2019-06-17 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1523980800, endDate = 1560700800")


rebuilt event_metrics rows: 58
unique events now: 5

Coverage matrix (1=has CAR for event):


event_id,1,2,3,4,5
ticker,,,,,
1101.TW,1.0,1.0,1.0,1.0,1.0
2317.TW,1.0,1.0,1.0,1.0,1.0
2330.TW,1.0,1.0,1.0,1.0,1.0
2404.TW,1.0,1.0,1.0,1.0,1.0
2454.TW,1.0,1.0,1.0,1.0,1.0
2850.TW,1.0,1.0,1.0,1.0,1.0
2881.TW,1.0,1.0,1.0,1.0,1.0
2882.TW,1.0,1.0,1.0,1.0,1.0
5536.TWO,1.0,1.0,1.0,1.0,1.0



Top by avg 5-day CAR (zero-impute, denominator = all post-2018 events):


,ticker,avg_car_0_to_5,denom_car_0_to_5,n_obs
0,2404.TW,0.050939,5,5
1,2882.TW,0.010985,5,5
2,2850.TW,0.009530,5,5
3,2881.TW,0.004422,5,5
4,6505.TW,0.002470,5,5
5,8926.TW,0.002206,5,5
6,1101.TW,-0.005392,5,5
7,2330.TW,-0.008071,5,5
8,2454.TW,-0.010827,5,5
9,6806.TW,-0.018247,5,3


Top by avg 5-day CAR (available-only, denominator = observed post-2018 events):


,ticker,avg_car_0_to_5,denom_car_0_to_5,n_obs
0,2404.TW,0.050939,5,5
1,2882.TW,0.010985,5,5
2,2850.TW,0.009530,5,5
3,2881.TW,0.004422,5,5
4,6505.TW,0.002470,5,5
5,8926.TW,0.002206,5,5
6,1101.TW,-0.005392,5,5
7,2330.TW,-0.008071,5,5
8,2454.TW,-0.010827,5,5
9,6806.TW,-0.030412,3,3


In [80]:
start = "2018-01-01"
end = dt.date.today().isoformat()
px = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)["Close"]

# Compute daily returns
rets = px.pct_change().dropna(how="all")

# Drop columns with too many NaNs and fill small gaps
valid_cols = [c for c in rets.columns if rets[c].notna().sum() > len(rets)*0.8]
rets = rets[valid_cols].fillna(0.0)

print("Returns shape:", rets.shape)
print("Tickers included:", list(rets.columns))

Returns shape: (1884, 2)
Tickers included: ['6505.TW', '8926.TW']


In [81]:
CAR_WINDOW = "car_0_to_5"        # {"car_0_to_3","car_0_to_5","car_0_to_10"}
AVERAGE_MODE = "available"       # "available" or "zero_impute"
p_event = 0.10                   # probability of an earthquake within CAR window
target_car_exposure = 0.00       # dot(w, avg CAR). 0=neutral; >0 benefit; <0 hedge
objective = "Sharpe"             # "Sharpe", "MinRisk", "Utility"
long_only = True                 # True = [0,1] weights; False = allow shorts
rf = 0.00                        # risk-free for Sharpe

def avg_car_by_ticker(event_metrics: pd.DataFrame,
                      earthquakes_df: pd.DataFrame,
                      window: str = "car_0_to_5",
                      mode: str = "available") -> pd.Series:
    assert window in {"car_0_to_3","car_0_to_5","car_0_to_10"}
    if earthquakes_df is None or earthquakes_df.empty:
        raise ValueError("earthquakes_df is empty. Provide your earthquakes list (e.g., earthquakes_recent).")
    em = event_metrics[event_metrics["event_id"].isin(earthquakes_df["event_id"])].copy()
    if em.empty:
        raise ValueError("event_metrics has no rows for provided earthquakes_df. Rebuild event_metrics first.")

    all_tickers = sorted(em["ticker"].unique())
    all_events  = list(earthquakes_df["event_id"])
    wide = (
        em.set_index(["ticker","event_id"])[[window]]
          .reindex(pd.MultiIndex.from_product([all_tickers, all_events], names=["ticker","event_id"]))
    )
    if mode == "available":
        return wide.groupby(level="ticker").mean()[window]
    elif mode == "zero_impute":
        return wide.fillna(0).groupby(level="ticker").mean()[window]
    else:
        raise ValueError("mode must be 'available' or 'zero_impute'")

# Use your earthquakes subset if defined, else all
_eq_df = earthquakes_recent.copy() if "earthquakes_recent" in locals() else (
         earthquakes.copy()        if "earthquakes" in locals()        else None)
if _eq_df is None:
    raise RuntimeError("Please define `earthquakes_recent` or `earthquakes` before running this cell.")

car_sensitivity = avg_car_by_ticker(event_metrics, _eq_df, window=CAR_WINDOW, mode=AVERAGE_MODE)

if "rets" not in locals():
    tickers = sorted(event_metrics["ticker"].unique())
    start = "2018-01-01"
    end   = dt.date.today().isoformat()
    px = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)["Close"]
    if isinstance(px, pd.Series):
        px = px.to_frame()
    rets = px.pct_change().dropna(how="all")
    # Drop sparse columns & fill tiny gaps
    valid_cols = [c for c in rets.columns if rets[c].notna().sum() > len(rets)*0.80]
    rets = rets[valid_cols].fillna(0.0)

# Helpers
def ensure_single_level_cols(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        lvl0 = df.columns.get_level_values(0)
        if "Close" in set(lvl0):
            df = df.xs("Close", axis=1, level=0)
        else:
            df.columns = ["_".join([str(x) for x in tup if x is not None])
                          for tup in df.columns.to_flat_index()]
    return df

# Normalize rets columns, dedupe, clean
rets = ensure_single_level_cols(rets).copy()
if "SPY_SPY" in rets.columns and "SPY" in rets.columns:
    rets = rets.drop(columns=["SPY_SPY"])
elif "SPY_SPY" in rets.columns:
    rets = rets.rename(columns={"SPY_SPY": "SPY"})
rets = rets.loc[:, ~rets.columns.duplicated()].copy()
rets = rets.astype(float).replace([np.inf, -np.inf], np.nan).dropna(how="any")
rets = rets.loc[:, rets.var() > 0]

# Add SPY if missing (baseline neutral to Taiwan quakes)
if "SPY" not in rets.columns:
    start = rets.index.min().strftime("%Y-%m-%d") if len(rets) else "2017-01-01"
    end   = dt.date.today().isoformat()
    px_spy = yf.download("SPY", start=start, end=end, auto_adjust=True, progress=False)
    spy_close = px_spy[["Close"]].rename(columns={"Close": "SPY"})
    rets_spy = spy_close.pct_change()
    rets_spy = ensure_single_level_cols(rets_spy)
    rets = pd.concat([rets, rets_spy], axis=1, join="inner").dropna(how="any")

# Align CAR vector; set SPY neutral
car_sensitivity = car_sensitivity.copy()
if "SPY_SPY" in car_sensitivity.index and "SPY" in rets.columns:
    car_sensitivity = car_sensitivity.drop(index="SPY_SPY")
car_sensitivity = car_sensitivity.reindex(rets.columns).fillna(0.0)
if "SPY" in car_sensitivity.index:
    car_sensitivity.loc["SPY"] = 0.0

print("Final columns going into Riskfolio:", list(rets.columns))
print("Final CAR index (aligned):", list(car_sensitivity.index))
print("Any NaNs left in rets?", rets.isna().any().any())

# Build portfolio + optimization (hard -> soft fallback)
def _build_port(returns: pd.DataFrame) -> rp.Portfolio:
    port = rp.Portfolio(returns=returns)
    # Use sample covariance while stabilizing
    port.assets_stats(method_mu="hist", method_cov="hist")
    return port

def _set_bounds(port: rp.Portfolio, n: int, long_only: bool):
    if long_only:
        port.lowerlng = np.zeros(n); port.upperlng = np.ones(n)
    else:
        port.lowerlng = -np.ones(n); port.upperlng =  np.ones(n)

def hard_optimize(rets: pd.DataFrame, cs: pd.Series, target: float, long_only: bool, objective: str, rf: float):
    port = _build_port(rets)
    _set_bounds(port, rets.shape[1], long_only)
    B = cs.values.reshape(1, -1)
    b = np.array([float(target)], dtype=float)
    weights = port.optimization(model="Classic", obj=objective, rf=rf, B=B, b=b)
    return pd.Series(np.ravel(weights), index=rets.columns, name="weight")

def soft_tilt_optimize(rets: pd.DataFrame, cs: pd.Series, lam: float, long_only: bool, objective: str, rf: float):
    port = _build_port(rets)
    port.mu = port.mu + lam * cs.values
    _set_bounds(port, rets.shape[1], long_only)
    weights = port.optimization(model="Classic", obj=objective, rf=rf)
    return pd.Series(np.ravel(weights), index=rets.columns, name="weight")

mode_used = "HARD (B@w=b)"
try:
    w = hard_optimize(rets, car_sensitivity, target_car_exposure, long_only, objective, rf)
except Exception:
    print("⚠️ Hard equality appears infeasible. Falling back to soft tilt.\n"
          "   Tips: allow shorts or set a small target with the CAR sign (e.g., 0.001).")
    w = soft_tilt_optimize(rets, car_sensitivity, lam=2.0, long_only=long_only, objective=objective, rf=rf)
    mode_used = "SOFT TILT (mu += λ·CAR)"


# Report weights
w = w[w.abs() > 1e-8]
achieved_exposure = float(np.dot(w.reindex(car_sensitivity.index).fillna(0.0), car_sensitivity.values))
expected_shock_contribution = float(p_event * achieved_exposure)

print(f"\n== Optimal weights [{mode_used}] ==")
display(w.sort_values(ascending=False).to_frame("Weight"))

print(f"Target CAR exposure:     {target_car_exposure:.6f}  (hard target)")
print(f"Achieved CAR exposure:   {achieved_exposure:.6f}")
print(f"p(event) over window:    {p_event:.2%}")
print(f"Expected shock contrib.: {expected_shock_contribution:.6f}   (p_event × exposure)")

# Quick annualized stats
ann_mu  = (1 + rets.mean()).pow(252) - 1
ann_cov = rets.cov() * 252
exp_ret = float((ann_mu.reindex(w.index) * w).sum())
exp_vol = float(np.sqrt(np.dot(w.values, np.dot(ann_cov.loc[w.index, w.index].values, w.values))))
sharpe  = exp_ret / exp_vol if exp_vol > 0 else np.nan
print(f"\nAnnualized: Return={exp_ret:.2%}, Vol={exp_vol:.2%}, Sharpe≈{sharpe:.2f}")

Final columns going into Riskfolio: ['6505.TW', '8926.TW', 'SPY_SPY']
Final CAR index (aligned): ['6505.TW', '8926.TW', 'SPY_SPY']
Any NaNs left in rets? False
⚠️ Hard equality appears infeasible. Falling back to soft tilt.
   Tips: allow shorts or set a small target with the CAR sign (e.g., 0.001).

== Optimal weights [SOFT TILT (mu += λ·CAR)] ==


/opt/conda/lib/python3.13/site-packages/cvxpy/reductions/solvers/solving_chain_utils.py:30: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


,Weight
8926.TW,0.651848
6505.TW,0.274711
SPY_SPY,0.073441


Target CAR exposure:     0.000000  (hard target)
Achieved CAR exposure:   0.002117
p(event) over window:    10.00%
Expected shock contrib.: 0.000212   (p_event × exposure)

Annualized: Return=9.96%, Vol=17.80%, Sharpe≈0.56
